<!-- nav -->
[← 2 · Finding what matches](02_Finding_What_Matches.ipynb) · [Home](../../README.md) · [4 · Rules that take arguments →](04_Rules_That_Take_Arguments.ipynb)

# 3 · Links as rules

**The problem.** Some rules are about links, not values: every contact has a phone (mandatory), no contact has a
pager (forbidden), most contacts have a phone (possible). And some rules are about the whole address book, not one
contact: "everyone has a phone". These need quantifiers over a schema's objects in the store.

This case study writes mandatory and forbidden links, and statements about the whole store, in the predicate algebra.

In [1]:
import { Expressions as E } from "@mbse/expressions";
import { Predicates, Validators } from "@mbse/patterns";
import { Contact, Directory, Phone, book, listed } from "./toolkit.ts";

const store = book();
const ann = store.Contact().name("Ann").phones((e: any) => e.label("home").phone((p: any) => p.number("1"))).create();
const bob = store.Contact().name("Bob").create();
listed(store, ann, bob);
const [c, p] = [E.variable("c"), E.variable("p")];

## Step 1: a quantifier

`Predicates.Exists(...)` says some object of a schema satisfies its conditions; `Forall(...)` says every one does. Each
is built by a builder that declares its symbols and conditions, as a predicate's does:

In [2]:
const HasAPhone = Predicates.Exists((q) => q.symbols({ p: Phone }).requires(Predicates.Contains(c.phones, (e) => e.phone.eq(p))));
const Mandatory = new Predicates.OfPredicate.Builder().name("Mandatory").symbols({ c: Contact }).requires(HasAPhone).create();
const Forbidden = new Predicates.OfPredicate.Builder().name("NoPhone").symbols({ c: Contact }).forbids(HasAPhone).create();
console.log(Validators.Validate(store, [Mandatory, Forbidden]).Reachable(Directory, store.singleton("book.Directory")));

[
  "c=Contact#2: 'Mandatory' does not hold",
  "c=Contact#1: 'NoPhone' does not hold"
]


Bob breaks the mandatory link; Ann breaks the forbidden one. `.forbids(...)` adds the negation of a condition.

## Step 2: a statement about the whole store

A predicate without symbols isn't about any one object: it's a statement about the store, checked once:

In [3]:
const EveryoneHasAPhone = new Predicates.OfPredicate.Builder().name("EveryoneHasAPhone")
  .requires(Predicates.Forall((q) => q.symbols({ c: Contact }).requires(HasAPhone))).create();
console.log(Validators.Validate(store, [EveryoneHasAPhone])(Contact, ann));

[ "the store: 'EveryoneHasAPhone' does not hold" ]


`Forall` ranges over the contacts in the store's data, its **extent**: what the directory reaches. Give Bob a phone
and the statement holds:

In [4]:
store.Contact(bob).phones((e: any) => e.label("work").phone((q: any) => q.number("2"))).update();
console.log(Validators.Validate(store, [EveryoneHasAPhone])(Contact, ann));

[]


## Step 3: evaluating terms of the algebra

`new Predicates.Evaluator(store).run(...)` evaluates any expression of the algebra over the store, with variables you
give:

In [5]:
const evaluate = new Predicates.Evaluator(store);
console.log(evaluate.run(HasAPhone, { c: ann }), evaluate.run(Predicates.Forall((q) => q.symbols({ c: Contact }).requires(c.has("age")))));

true false


Nobody's age is on file, so "every contact has an age" is false, not unknown: `has` asks whether a property is
present, and that has an answer.

## Why it works this way

- **Quantifiers range over the store's data.** An object you built but never listed isn't in the extent, as with
  queries: rules about the store are about what the store holds.
- **Links are relations, so link rules are `Contains`.** mbse-schemas has no arrays of links; a contact's phones are
  entries of a relation. `Contains` reads those entries, and a query plans it as a hop.
- **"Possible" is a weight, not a truth.** "Most contacts have a phone" doesn't fail any one contact; it says how
  contacts are distributed. Case study 5 writes it.

<!-- nav -->
[← 2 · Finding what matches](02_Finding_What_Matches.ipynb) · [Home](../../README.md) · [4 · Rules that take arguments →](04_Rules_That_Take_Arguments.ipynb)